# `soc-pipeline` Quickstart — Earthquake SOC validation

This notebook walks through the canonical use-case of `soc-pipeline`:
validating a Gutenberg-Richter-like power-law on earthquake magnitudes
with a pre-registered acceptance band.

The full pipeline runs end-to-end in ~10 cells:

1. Install + import
2. Synthesise a Gutenberg-Richter catalogue (b≈1.0 → Clauset α≈2.0)
3. One-call `validate()` with a pre-registered band
4. Inspect Verdict
5. Low-level: standalone `fit_clauset_powerlaw` for diagnostics
6. Bootstrap CI (200 resamples)
7. Synthetic null controls
8. Empirical CCDF plot vs theoretical power-law
9. Negative-control: feed an exponential, expect FAIL
10. Summary table

All cells are deterministic (seeded RNGs).

## 1. Install + import

In [ ]:
# pip install soc-pipeline[notebooks]
import numpy as np
from soc_pipeline import (
    validate,
    fit_clauset_powerlaw,
    bootstrap_ci,
    synthetic_null,
    empirical_ccdf,
    __version__ as SOC_VERSION,
)

print(f'soc-pipeline {SOC_VERSION}')

## 2. Synthesise a Gutenberg-Richter catalogue

GR law: $\log_{10} N(M{\ge}m) = a - bm$. With b≈1.0, the energy-released
distribution follows a power-law with Clauset exponent α≈2.0.

We draw 5000 magnitudes ≥ 3 from a tail-only Pareto with the correct slope.

In [ ]:
rng = np.random.default_rng(seed=2026)
# Clauset alpha = 2.0 → numpy.random.pareto shape = alpha - 1 = 1.0
magnitudes = (rng.pareto(1.0, size=5000) + 1.0) * 3.0  # min M = 3.0
print(f'N events       = {magnitudes.size}')
print(f'Min magnitude  = {magnitudes.min():.2f}')
print(f'Max magnitude  = {magnitudes.max():.2f}')
print(f'Median         = {float(np.median(magnitudes)):.2f}')

## 3. One-call `validate()` with a pre-registered band

Pre-registered band: α ∈ [1.8, 2.2] (literature consensus for global GR).

In [ ]:
verdict = validate(
    magnitudes,
    label='earthquake_magnitudes',
    expected_band=(1.8, 2.2),
)
print(f'verdict     : {verdict.verdict}')
print(f'alpha       : {verdict.alpha:.3f}  [{verdict.alpha_ci_lo:.3f}, {verdict.alpha_ci_hi:.3f}]')
print(f'xmin        : {verdict.xmin:.3f}')
print(f'n_tail      : {verdict.n_tail}')
print(f'KS distance : {verdict.ks_distance:.4f}')
print(f'in_band     : {verdict.in_band}')
print(f'reason      : {verdict.reason}')

## 4. Inspect the full `Verdict` dataclass

Every numeric field is finite and serialisable.

In [ ]:
from dataclasses import asdict
import json
print(json.dumps(asdict(verdict), indent=2, default=str))

## 5. Standalone fit (low-level API)

In [ ]:
fit = fit_clauset_powerlaw(magnitudes, name='earthquake_magnitudes')
print(f'alpha               : {fit.alpha:.3f} ± {fit.sigma:.3f}')
print(f'xmin                : {fit.xmin:.3f}')
print(f'n_tail / n_total    : {fit.n_tail} / {fit.n_total}')
print(f'KS                  : {fit.ks_statistic:.4f}')
print(f'vs lognormal R, p   : {fit.vs_lognormal_R:.3f}, {fit.vs_lognormal_p:.3f}')
print(f'vs exponential R, p : {fit.vs_exponential_R:.3f}, {fit.vs_exponential_p:.3f}')
print(f'PL vs LN winner     : {fit.vs_powerlaw_lognormal_winner}')

## 6. Bootstrap confidence interval (200 resamples)

In [ ]:
ci = bootstrap_ci(magnitudes, n_boot=200, seed=42)
print(f'mean alpha   : {ci.alpha_mean:.3f}')
print(f'median alpha : {ci.alpha_median:.3f}')
print(f'std alpha    : {ci.alpha_std:.4f}')
print(f'95% CI       : [{ci.ci_low:.3f}, {ci.ci_high:.3f}]')
print(f'n succeeded  : {ci.n_boot_succeeded}/200')

## 7. Synthetic null controls

`synthetic_null()` generates 3 explicitly-not-power-law samples (lognormal,
exponential, gaussian) and asserts the validator rejects them all.
Use this to convince a reviewer that the pipeline is not biased toward FAILing power-law.

In [ ]:
nulls = synthetic_null(n=5000, seed=42)
for name, case in nulls.items():
    print(f'  {name:14s}  alpha={case.fit.alpha!s:8}  rejected? {case.correctly_rejected}')

## 8. Empirical CCDF

If `matplotlib` is installed (extra `[notebooks]`), this plots the empirical
CCDF on log-log axes. Otherwise prints a tabular preview.

In [ ]:
grid, ccdf = empirical_ccdf(magnitudes, n_points=200)
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.loglog(grid, ccdf, 'k.', label='empirical')
    plt.xlabel('event size')
    plt.ylabel('P(X > x)')
    plt.title(f'CCDF — α≈{verdict.alpha:.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed; install soc-pipeline[notebooks])')
    for g, c in list(zip(grid, ccdf))[::20]:
        print(f'  x={g:8.2f}  P(X>x)={c:.4f}')

## 9. Negative control — exponential should FAIL

In [ ]:
exp_sample = rng.exponential(scale=10.0, size=5000)
neg = validate(exp_sample, label='exponential_negative_control', expected_band=(1.8, 2.2))
print(f'verdict : {neg.verdict}')
print(f'alpha   : {neg.alpha:.3f}')
print(f'reason  : {neg.reason}')

## 10. Summary table

In [ ]:
rows = [
    ('earthquake (true power-law)', verdict),
    ('exponential (negative control)', neg),
]
fmt = '{:35s}  {:12s}  alpha={:6.3f}  band? {:5s}'
for label, v in rows:
    print(fmt.format(label, v.verdict, v.alpha, str(v.in_band)))